# Rolling Window Runner (Minimal)

This notebook has 3 parts:

1. Description (this section)
2. Control panel to choose HAR/LSTM variants and covariables
3. Single execution cell that runs `src/evaluation/eval_rolling_window.py`

Use the control panel only, then run the final cell.


In [8]:
%pip install -e ..

from itertools import product
import importlib

import src.evaluation.eval_rolling_window as erw
from src.config.paths import INTERIM_DAILY_RV_DIR
from src.models.types import HARConfig
from src.models.LSTM import LSTMConfig

# Force reload so notebook uses latest source changes (ratio_event / lambda_t labels).
erw = importlib.reload(erw)
RollingWindowConfig = erw.RollingWindowConfig
RollingWindowEvaluator = erw.RollingWindowEvaluator

# -----------------------------------------------------------------------------
# CONTROL PANEL
# -----------------------------------------------------------------------------

# Coins to include in both HAR and LSTM pipelines
COIN_SELEC = ["BTC", "ETH"] # ["BTC", "ETH"]

# Directory with per-coin daily RV parquet files (<COIN>_rv.parquet).
DAILY_RV_DIR = INTERIM_DAILY_RV_DIR

# Covariables to include in both HAR and LSTM pipelines
# Available options: None, "ratio_event", "exceedance_down", "lambda_t",
# or any list combining those daily covariates.
COVARIABLES_SELEC = [
    None,
    "ratio_event",
    "exceedance_down",
    "lambda_t",
    ["ratio_event", "exceedance_down"],
    ["exceedance_down", "lambda_t"],
    ["ratio_event", "exceedance_down", "lambda_t"],
]

# Rolling window configuration for evaluation
TRAIN_WINDOW = 1200
TEST_WINDOW = 300
STEP = 300
LSTM_VAL_WINDOW = 300

# LSTM architecture / optimizer search space
HEAD_OPTIONS = ["linear"]                     # fixed best family
DEPTH_OPTIONS = [1]                            # fixed best depth
OPTIMIZER_OPTIONS = ["adam"]                  # fixed best optimizer
OUTPUT_ACTIVATION_OPTIONS = ["identity"]      # fixed best output activation

# Hyperparameter tuning search space (new)
HIDDEN_SIZE_OPTIONS = [32, 64]     # LSTM hidden size options
EARLY_STOPPING_OPTIONS = [True]         # False=no ES, True=ES on
EARLY_STOPPING_PATIENCE = 30
EARLY_STOPPING_MIN_DELTA = 1e-9

# Shared model/training configuration
CRITERION = "mse"                              # mse / qlike
ACTIVATION = "tanh"                            # relu | gelu | tanh (MLP internal activation)
LOOKBACK_DAYS = 30
HEAD_HIDDEN_SIZE = 32                           # only used when head="mlp"
REPRESENTATION = "last_hidden"                 # currently fixed in implementation

LR = 1e-3
BATCH_SIZE = 32
EPOCHS = 500


# HAR mode / variant
HAR_VARIANTS = [
    HARConfig(name="HAR-baseline"),
]

# LSTM mode / variants generated from all combinations.
# name format: LSTM-hs<hidden_size>
LSTM_VARIANTS = [
    LSTMConfig(
        name=f"LSTM-hs{hidden_size}",
        head=head,
        depth=depth,
        lookback_days=LOOKBACK_DAYS,
        representation=REPRESENTATION,
        head_hidden_size=HEAD_HIDDEN_SIZE,
        activation=ACTIVATION,
        output_activation=output_activation,
        criterion=CRITERION,
        optimizer=optimizer,
        hidden_size=hidden_size,
        lr=LR,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        patience=EARLY_STOPPING_PATIENCE if use_early_stopping else 0,
        min_delta=EARLY_STOPPING_MIN_DELTA if use_early_stopping else 0.0,
    )
    for head, depth, optimizer, output_activation, hidden_size, use_early_stopping in product(
        HEAD_OPTIONS,
        DEPTH_OPTIONS,
        OPTIMIZER_OPTIONS,
        OUTPUT_ACTIVATION_OPTIONS,
        HIDDEN_SIZE_OPTIONS,
        EARLY_STOPPING_OPTIONS,
    )
]

print("Control panel ready")
print(f"COIN={COIN_SELEC}")
print(f"daily_rv_dir={DAILY_RV_DIR}")
print(f"Covariables={COVARIABLES_SELEC}")
print(f"HAR variants={len(HAR_VARIANTS)}")
print(f"LSTM variants={len(LSTM_VARIANTS)}")
print("LSTM names:")
for cfg in LSTM_VARIANTS:
    print(f"  - {cfg.name}")


Obtaining file:///C:/Users/diego/OneDrive%20-%20Universidad%20T%C3%A9cnica%20Federico%20Santa%20Mar%C3%ADa/%2800%29-Documents/%2801%29%20Proyectos/%2803%29%20Github/Finanzas/Hawkes%20Enhanced%20RV%20Forecast
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for hawkes-enhanced-rv-forecast (pyproject.toml): started
  Building editable for hawkes-enhanced-rv-forecast (pyproject.toml): finished with status 'done'
  Created wheel for hawkes-enhanced-rv-forecast: filename=hawkes_enhanced_rv_forecast-0.1.0-0.editable-py3-none-any.whl 

In [9]:
# -----------------------------------------------------------------------------
# RUN
# -----------------------------------------------------------------------------
for coin in COIN_SELEC:
    for covariables in COVARIABLES_SELEC:
        cfg = RollingWindowConfig(
            train_window=TRAIN_WINDOW,
            test_window=TEST_WINDOW,
            step=STEP,
            lstm_val_window=LSTM_VAL_WINDOW,
            feature_cols=covariables,
            verbose=True,
        )

        evaluator = RollingWindowEvaluator(
            coin=coin,
            rolling_config=cfg,
            har_configs=HAR_VARIANTS,
            lstm_configs=LSTM_VARIANTS,
            daily_rv_dir=DAILY_RV_DIR,
        )

        preds, summary = evaluator.run()

        print("\n=== Summary ===")
        print(summary)

        summary = summary.sort_values(["model_name", "qlike", "rmse"]).reset_index(drop=True)
        display(summary)

Loaded daily RV for BTC: 3059 rows from C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\data\interim\daily_rv\BTC_rv.parquet
Feature matrices aligned: 2596 rows | train=1200, test=300, step=300, n_windows=4
Results directory: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\results\coin=BTC
[HAR:HAR-baseline] feature_tag=base | ready_windows=0/4
  [HAR:HAR-baseline] window 0: running (start_idx=0, train=1200, test=300)
  [HAR:HAR-baseline] window 0: saved 300 rows -> predictions__kind=HAR__model=HAR-baseline__features=base.parquet
  [HAR:HAR-baseline] window 1: running (start_idx=300, train=1200, test=300)
  [HAR:HAR-baseline] window 1: saved 300 rows -> predictions__kind=HAR__model=HAR-baseline__features=base.parquet
  [HAR:HAR-baseline] window 2: running (start_idx=600, train=1200, test=300)


,coin,model_name,model_kind,feature_tag,n_forecasts,n_windows,mse,rmse,mae,qlike
0,BTC,HAR-baseline,HAR,base,1200,4,0.000111,0.010518,0.007167,0.333125
1,BTC,LSTM-hs32,LSTM,base,1200,4,0.000147,0.012128,0.008480,0.430084
2,BTC,LSTM-hs64,LSTM,base,1200,4,0.000153,0.012375,0.008815,0.431066


Loaded daily RV for BTC: 3059 rows from C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\data\interim\daily_rv\BTC_rv.parquet
Feature matrices aligned: 2596 rows | train=1200, test=300, step=300, n_windows=4
Results directory: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\results\coin=BTC
[HAR:HAR-baseline] feature_tag=base+ratio_event | ready_windows=0/4
  [HAR:HAR-baseline] window 0: running (start_idx=0, train=1200, test=300)
  [HAR:HAR-baseline] window 0: saved 300 rows -> predictions__kind=HAR__model=HAR-baseline__features=base_ratio_event.parquet
  [HAR:HAR-baseline] window 1: running (start_idx=300, train=1200, test=300)
  [HAR:HAR-baseline] window 1: saved 300 rows -> predictions__kind=HAR__model=HAR-baseline__features=base_ratio_event.parquet
  [HAR:HAR-baseline] window 2: running (s

,coin,model_name,model_kind,feature_tag,n_forecasts,n_windows,mse,rmse,mae,qlike
0,BTC,HAR-baseline,HAR,base+ratio_event,1200,4,0.000109,0.010418,0.007118,0.327857
1,BTC,LSTM-hs32,LSTM,base+ratio_event,1200,4,0.000153,0.012367,0.008821,0.443125
2,BTC,LSTM-hs64,LSTM,base+ratio_event,1200,4,0.000145,0.012030,0.008362,0.443617


Loaded daily RV for BTC: 3059 rows from C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\data\interim\daily_rv\BTC_rv.parquet
Feature matrices aligned: 2596 rows | train=1200, test=300, step=300, n_windows=4
Results directory: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\results\coin=BTC
[HAR:HAR-baseline] feature_tag=base+exceedance_down | ready_windows=0/4
  [HAR:HAR-baseline] window 0: running (start_idx=0, train=1200, test=300)
  [HAR:HAR-baseline] window 0: saved 300 rows -> predictions__kind=HAR__model=HAR-baseline__features=base_exceedance_down.parquet
  [HAR:HAR-baseline] window 1: running (start_idx=300, train=1200, test=300)
  [HAR:HAR-baseline] window 1: saved 300 rows -> predictions__kind=HAR__model=HAR-baseline__features=base_exceedance_down.parquet
  [HAR:HAR-baseline] window 2

...

,coin,model_name,model_kind,feature_tag,n_forecasts,n_windows,mse,rmse,mae,qlike
0,ETH,HAR-baseline,HAR,base+ratio_event+exceedance_down,1200,4,0.000163,0.012763,0.008751,0.281678
1,ETH,LSTM-hs32,LSTM,base+ratio_event+exceedance_down,1200,4,0.000236,0.015373,0.010828,0.416148
2,ETH,LSTM-hs64,LSTM,base+ratio_event+exceedance_down,1200,4,0.000243,0.015594,0.011208,0.426212


Loaded daily RV for ETH: 3059 rows from C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\data\interim\daily_rv\ETH_rv.parquet
Feature matrices aligned: 2580 rows | train=1200, test=300, step=300, n_windows=4
Results directory: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\results\coin=ETH
[HAR:HAR-baseline] feature_tag=base+exceedance_down+lambda_t | ready_windows=0/4
  [HAR:HAR-baseline] window 0: running (start_idx=0, train=1200, test=300)
  [HAR:HAR-baseline] window 0: saved 300 rows -> predictions__kind=HAR__model=HAR-baseline__features=base_exceedance_down_lambda_t.parquet
  [HAR:HAR-baseline] window 1: running (start_idx=300, train=1200, test=300)
  [HAR:HAR-baseline] window 1: saved 300 rows -> predictions__kind=HAR__model=HAR-baseline__features=base_exceedance_down_lambda_t.parquet
  

,coin,model_name,model_kind,feature_tag,n_forecasts,n_windows,mse,rmse,mae,qlike
0,ETH,HAR-baseline,HAR,base+exceedance_down+lambda_t,1200,4,0.000167,0.012904,0.008939,0.288605
1,ETH,LSTM-hs32,LSTM,base+exceedance_down+lambda_t,1200,4,0.000238,0.015413,0.010722,0.429732
2,ETH,LSTM-hs64,LSTM,base+exceedance_down+lambda_t,1200,4,0.000235,0.015324,0.010737,0.414630


Loaded daily RV for ETH: 3059 rows from C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\data\interim\daily_rv\ETH_rv.parquet
Feature matrices aligned: 2580 rows | train=1200, test=300, step=300, n_windows=4
Results directory: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\results\coin=ETH
[HAR:HAR-baseline] feature_tag=base+ratio_event+exceedance_down+lambda_t | ready_windows=0/4
  [HAR:HAR-baseline] window 0: running (start_idx=0, train=1200, test=300)
  [HAR:HAR-baseline] window 0: saved 300 rows -> predictions__kind=HAR__model=HAR-baseline__features=base_ratio_event_exceedance_down_lambda_t.parquet
  [HAR:HAR-baseline] window 1: running (start_idx=300, train=1200, test=300)
  [HAR:HAR-baseline] window 1: saved 300 rows -> predictions__kind=HAR__model=HAR-baseline__features=base_ratio_event

,coin,model_name,model_kind,feature_tag,n_forecasts,n_windows,mse,rmse,mae,qlike
0,ETH,HAR-baseline,HAR,base+ratio_event+exceedance_down+lambda_t,1200,4,0.000166,0.012885,0.008943,0.286454
1,ETH,LSTM-hs32,LSTM,base+ratio_event+exceedance_down+lambda_t,1200,4,0.000237,0.015398,0.010864,0.418598
2,ETH,LSTM-hs64,LSTM,base+ratio_event+exceedance_down+lambda_t,1200,4,0.000245,0.015646,0.011129,0.440235
